# ✈️ Travel Receipt Automation
**Scans Gmail for Amtrak, Marriott, and Uber receipts → saves to Google Drive → exports Excel/CSV**

---
### How to use this notebook
1. Run cells **in order** from top to bottom (Shift+Enter or click ▶)
2. Only edit the **⚙️ Configuration** cell — set your date range
3. The Excel file will appear in your **VibeCodingExperiments/TravelReceipts** Drive folder automatically

> **First run only:** Step 2 will show a Google sign-in popup — click Allow.

## 📦 Step 1 — Install Dependencies
Run once. Takes about 30 seconds.

In [28]:
!pip install -q pdfplumber beautifulsoup4 openpyxl python-dateutil lxml
print('✅ All dependencies installed.')

✅ All dependencies installed.


## 🔐 Step 2 — Authenticate with Google
A popup will appear asking you to sign in. Use the same Gmail account where your receipts are.

In [29]:
from google.colab import auth
auth.authenticate_user()

import google.auth
SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/drive'
]
creds, _ = google.auth.default(scopes=SCOPES)
print('✅ Authenticated successfully!')

✅ Authenticated successfully!


## ⚙️ Step 3 — Configuration
**Edit the dates below** to set the month range you want to scan. Format: `YYYY-MM`

In [30]:
# ── Date range to scan ───────────────────────────────────────────────────────
START_MONTH = '2026-01'   # e.g. '2026-01' = January 2026
END_MONTH   = '2026-04'   # e.g. '2026-04' = April 2026 (inclusive)

# ── Vendors to include ───────────────────────────────────────────────────────
VENDORS = ['amtrak', 'marriott', 'uber']   # remove any you don't need

# ── Google Drive target folder ───────────────────────────────────────────────
# My Drive > VibeCodingExperiments > TravelReceipts
DRIVE_FOLDER_ID = '1dh4Va3pawmNb8ZBuaViEa_4-uhItLXl0'

# ── Gmail search queries (tuned to your actual receipt senders) ──────────────
GMAIL_QUERIES = {
    'amtrak':   'from:etickets@amtrak.com subject:"SALES RECEIPT"',
    'marriott': '(from:marriott.com OR from:email.marriott.com) (subject:"receipt" OR subject:"folio" OR subject:"confirmation")',
    'uber':     'from:noreply@uber.com (subject:"trip" OR subject:"tipping" OR subject:"receipt")',
}

# ── Local working directory inside Colab ─────────────────────────────────────
DOWNLOAD_DIR = '/content/downloads'
OUTPUT_DIR   = '/content/output'

import os
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)
print(f'✅ Config ready. Scanning {START_MONTH} → {END_MONTH} for: {", ".join(VENDORS)}')

✅ Config ready. Scanning 2026-01 → 2026-04 for: amtrak, marriott, uber


## 📧 Step 4 — Gmail Scanner
Defines the class that searches Gmail and downloads receipt attachments.

In [31]:
import base64, re
from datetime import datetime, date
from typing import List, Dict, Optional
from email.utils import parsedate_to_datetime
import calendar
from googleapiclient.discovery import build


class GmailScanner:
    ATTACHMENT_MIME_WHITELIST = {'application/pdf', 'text/html', 'text/plain'}

    def __init__(self, credentials):
        self.service = build('gmail', 'v1', credentials=credentials)

    def download_receipts(self, start_month, end_month, download_dir, vendors=None, queries=None):
        after  = _first_day(start_month).strftime('%Y/%m/%d')
        before = _last_day(end_month).strftime('%Y/%m/%d')
        date_filter = f'after:{after} before:{before}'
        target_vendors = vendors or list(queries.keys())
        downloaded = []

        for vendor in target_vendors:
            base_query = (queries or {}).get(vendor)
            if not base_query:
                print(f'⚠️  No query for {vendor}, skipping.')
                continue
            full_query = f'{base_query} {date_filter}'
            print(f'\n📧 Searching {vendor.upper()} receipts…')
            print(f'   Query: {full_query}')
            messages = self._search(full_query)
            print(f'   Found {len(messages)} message(s).')
            for msg in messages:
                downloaded.extend(self._process(msg['id'], vendor, download_dir))

        return downloaded

    def _search(self, query):
        results, page_token = [], None
        while True:
            kwargs = {'userId': 'me', 'q': query, 'maxResults': 100}
            if page_token:
                kwargs['pageToken'] = page_token
            resp = self.service.users().messages().list(**kwargs).execute()
            results.extend(resp.get('messages', []))
            page_token = resp.get('nextPageToken')
            if not page_token:
                break
        return results

    def _process(self, msg_id, vendor, download_dir):
        msg     = self.service.users().messages().get(userId='me', id=msg_id, format='full').execute()
        headers = {h['name']: h['value'] for h in msg['payload']['headers']}
        subject = headers.get('Subject', '(no subject)')
        date_str= headers.get('Date', '')
        try:
            msg_date = parsedate_to_datetime(date_str).strftime('%Y-%m-%d')
        except Exception:
            msg_date = date_str
        print(f'  → [{vendor}] {msg_date}  {subject[:60]}')
        records = []
        self._extract_parts(msg['payload'], msg_id, vendor, subject, msg_date, download_dir, records)
        if not records:
            body = self._save_body(msg['payload'], msg_id, vendor, subject, msg_date, download_dir)
            if body:
                records.append(body)
        return records

    def _extract_parts(self, payload, msg_id, vendor, subject, msg_date, download_dir, records):
        mime = payload.get('mimeType', '')
        body = payload.get('body', {})
        if body.get('data') and mime in self.ATTACHMENT_MIME_WHITELIST and payload.get('filename'):
            filename   = payload['filename']
            local_path = self._save(base64.urlsafe_b64decode(body['data']), filename, msg_id, vendor, download_dir)
            records.append({'vendor': vendor, 'message_id': msg_id, 'subject': subject,
                            'date': msg_date, 'filename': filename, 'local_path': local_path, 'mime_type': mime})
        if body.get('attachmentId') and mime in self.ATTACHMENT_MIME_WHITELIST:
            filename = payload.get('filename') or f'{msg_id}.pdf'
            att = self.service.users().messages().attachments().get(
                userId='me', messageId=msg_id, id=body['attachmentId']).execute()
            local_path = self._save(base64.urlsafe_b64decode(att['data']), filename, msg_id, vendor, download_dir)
            records.append({'vendor': vendor, 'message_id': msg_id, 'subject': subject,
                            'date': msg_date, 'filename': filename, 'local_path': local_path, 'mime_type': mime})
        for part in payload.get('parts', []):
            self._extract_parts(part, msg_id, vendor, subject, msg_date, download_dir, records)

    def _save_body(self, payload, msg_id, vendor, subject, msg_date, download_dir):
        for mime_pref in ('text/html', 'text/plain'):
            data = _find_body_data(payload, mime_pref)
            if data:
                ext      = 'html' if mime_pref == 'text/html' else 'txt'
                filename = f'{vendor}_{msg_id}.{ext}'
                local_path = self._save(base64.urlsafe_b64decode(data), filename, msg_id, vendor, download_dir)
                return {'vendor': vendor, 'message_id': msg_id, 'subject': subject,
                        'date': msg_date, 'filename': filename, 'local_path': local_path, 'mime_type': mime_pref}
        return None

    @staticmethod
    def _save(data, filename, msg_id, vendor, directory):
        safe_name  = filename.replace('/', '_').replace('\\', '_')
        final_name = f'{vendor}_{msg_id}_{safe_name}'
        local_path = os.path.join(directory, final_name)
        with open(local_path, 'wb') as fh:
            fh.write(data)
        print(f'     💾 Saved: {final_name}')
        return local_path


def _first_day(ym):
    return datetime.strptime(ym, '%Y-%m').date().replace(day=1)

def _last_day(ym):
    d = datetime.strptime(ym, '%Y-%m').date()
    return d.replace(day=calendar.monthrange(d.year, d.month)[1])

def _find_body_data(payload, target_mime):
    if payload.get('mimeType') == target_mime:
        return payload.get('body', {}).get('data')
    for part in payload.get('parts', []):
        found = _find_body_data(part, target_mime)
        if found:
            return found
    return None

print('✅ GmailScanner ready.')

✅ GmailScanner ready.


## 🔍 Step 5 — Receipt Parsers
Defines parsers for Amtrak, Marriott, and Uber receipts.

In [32]:
import re, os
from typing import Dict, Optional
import pdfplumber
from bs4 import BeautifulSoup
from dateutil import parser as dparser


# ── Amtrak Parser ─────────────────────────────────────────────────────────────
class AmtrakParser:
    VENDOR = 'Amtrak'

    def parse(self, file_path):
        text = self._read(file_path)
        return {
            'vendor':        self.VENDOR,
            'date':          self._departure_date(text),
            'purchase_date': self._purchase_date(text),
            'amount':        self._amount(text),
            'description':   self._description(text),
            'reservation':   self._reservation(text),
            'ticket_number': self._ticket(text),
            'passenger':     self._passenger(text),
            'source_file':   file_path,
        }

    def _read(self, path):
        ext = os.path.splitext(path)[1].lower()
        if ext in ('.html', '.htm', '.txt'):
            with open(path, 'r', errors='replace') as f:
                raw = f.read()
            return BeautifulSoup(raw, 'lxml').get_text(separator='\n') if '<' in raw else raw
        try:
            import pdfplumber
            with pdfplumber.open(path) as pdf:
                return '\n'.join(p.extract_text() or '' for p in pdf.pages)
        except Exception:
            with open(path, 'r', errors='replace') as f:
                return f.read()

    def _departure_date(self, text):
        matches = re.findall(
            r'Depart\s+\d{1,2}:\d{2}\s+[AP]M,\s+\w+,\s+(\w+ \d{1,2},?\s*\d{4})', text, re.IGNORECASE)
        if matches:
            return _pd(matches[0])
        m = re.search(r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s*\d{4}\b', text, re.IGNORECASE)
        return _pd(m.group(0)) if m else None

    def _purchase_date(self, text):
        m = re.search(r'Purchased[:\s]+(\d{1,2}/\d{1,2}/\d{4})', text, re.IGNORECASE)
        return _pd(m.group(1)) if m else None

    def _amount(self, text):
        m = re.search(r'Total\s+Charged\s+by\s+Amtrak\s*\$?\s*([\d,]+\.\d{2})', text, re.IGNORECASE)
        if m: return float(m.group(1).replace(',', ''))
        m = re.search(r'\bTotal\b\s*\$?\s*([\d,]+\.\d{2})', text, re.IGNORECASE)
        if m: return float(m.group(1).replace(',', ''))
        amounts = re.findall(r'\$\s*([\d,]+\.\d{2})', text)
        return float(amounts[-1].replace(',', '')) if amounts else None

    def _description(self, text):
        m = re.search(r'([A-Za-z ,\-]+,\s*[A-Z]{2}\s+to\s+[A-Za-z ,\-]+(?:Station|Terminal|Airport)?[^\n]*)', text, re.IGNORECASE)
        if m:
            return re.sub(r'\s+to\s+', ' → ', m.group(1).strip(), count=1, flags=re.IGNORECASE)
        m = re.search(r'TRAIN\s+\d+[:\s]+(.+)', text, re.IGNORECASE)
        return m.group(1).strip() if m else 'Amtrak Trip'

    def _reservation(self, text):
        m = re.search(r'Reservation\s+Number\s*[-–:]\s*([A-Z0-9]{4,12})', text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    def _ticket(self, text):
        m = re.search(r'Ticket\s+Number\s*[-–:]?\s*(\d{6,20})', text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    def _passenger(self, text):
        m = re.search(r'Passengers?\s*\n+\s*([A-Z][a-z]+ [A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)', text)
        return m.group(1).strip() if m else None


# ── Marriott Parser ───────────────────────────────────────────────────────────
class MarriottParser:
    VENDOR = 'Marriott'

    def parse(self, file_path):
        text = self._read(file_path)
        check_in  = self._arrive(text)
        check_out = self._depart(text)
        return {
            'vendor':        self.VENDOR,
            'date':          check_in,
            'checkout_date': check_out,
            'nights':        self._nights(check_in, check_out),
            'amount':        self._amount(text),
            'description':   self._property(text),
            'guest_number':  self._guest_num(text),
            'room_number':   self._room(text),
            'bonvoy_number': self._bonvoy(text),
            'guest_name':    self._guest_name(text),
            'charge_summary':self._charges(text),
            'source_file':   file_path,
        }

    def _read(self, path):
        if os.path.splitext(path)[1].lower() in ('.html', '.htm'):
            with open(path, 'r', errors='replace') as f:
                return BeautifulSoup(f.read(), 'lxml').get_text(separator='\n')
        with pdfplumber.open(path) as pdf:
            return '\n'.join(p.extract_text() or '' for p in pdf.pages)

    def _arrive(self, text):
        m = re.search(r'Arrive\s+Date\s+(\d{1,2}-[A-Z]{3}-\d{2,4})', text, re.IGNORECASE)
        return _pd(m.group(1)) if m else None

    def _depart(self, text):
        m = re.search(r'Depart\s+Date\s+(\d{1,2}-[A-Z]{3}-\d{2,4})', text, re.IGNORECASE)
        return _pd(m.group(1)) if m else None

    def _amount(self, text):
        m = re.search(r'\*+\s*Total\s+([\d,]+\.\d{2})', text, re.IGNORECASE)
        if m: return float(m.group(1).replace(',', ''))
        amounts = re.findall(r'(?<!\-)\b([\d,]+\.\d{2})\b', text)
        return float(amounts[-1].replace(',', '')) if amounts else None

    def _property(self, text):
        for line in text.splitlines()[:30]:
            line = line.strip()
            if re.search(r'Marriott|Meridien|Courtyard|Sheraton|Westin|Renaissance|Autograph|Moxy|Aloft|Residence Inn|Fairfield', line, re.IGNORECASE):
                if len(line) > 5 and not re.search(r'copyright|reserved|bonvoy', line, re.IGNORECASE):
                    return line
        return 'Marriott Stay'

    def _guest_num(self, text):
        m = re.search(r'Guest\s+Number\s+(\d+)', text, re.IGNORECASE)
        return m.group(1) if m else None

    def _room(self, text):
        m = re.search(r'Room\s+Number\s+(\w+)', text, re.IGNORECASE)
        return m.group(1) if m else None

    def _bonvoy(self, text):
        m = re.search(r'(?:Marriott\s+)?Bonvoy\s+Number\s+(\w+)', text, re.IGNORECASE)
        return m.group(1) if m else None

    def _guest_name(self, text):
        m = re.search(r'^([A-Z]{2,}\s+[A-Z]{2,}(?:\s+[A-Z]{2,})?)\s*$', text, re.MULTILINE)
        return m.group(1).strip() if m else None

    def _charges(self, text):
        descs  = re.findall(r'(?:\d{1,2}-[A-Z]{3}-\d{2,4})\s+\w+\s+(.+?)\s+[\d,]+\.\d{2}', text)
        seen, unique = [], []
        for d in descs:
            if d.strip() and d.strip() not in seen:
                seen.append(d.strip()); unique.append(d.strip())
        return ' | '.join(unique) if unique else 'Hotel Stay'

    def _nights(self, ci, co):
        if not ci or not co: return None
        try: return (dparser.parse(co) - dparser.parse(ci)).days
        except: return None


# ── Uber Parser ───────────────────────────────────────────────────────────────
class UberParser:
    VENDOR = 'Uber'

    def parse(self, file_path):
        text, subject = self._read(file_path)
        return {
            'vendor':      self.VENDOR,
            'date':        self._date(text),
            'amount':      self._total(text),
            'trip_fare':   self._fare(text),
            'tip':         self._tip(text),
            'description': subject or self._desc(text),
            'passenger':   self._passenger(text),
            'source_file': file_path,
        }

    def _read(self, path):
        ext = os.path.splitext(path)[1].lower()
        if ext in ('.html', '.htm'):
            with open(path, 'r', errors='replace') as f:
                raw = f.read()
            soup    = BeautifulSoup(raw, 'lxml')
            title   = soup.find('title')
            subject = title.get_text(strip=True) if title else None
            return soup.get_text(separator='\n'), subject
        with open(path, 'r', errors='replace') as f:
            return f.read(), None

    def _date(self, text):
        m = re.search(r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{1,2},\s*\d{4}\b', text, re.IGNORECASE)
        return _pd(m.group(0)) if m else None

    def _total(self, text):
        m = re.search(r'\bTotal\b\s*\$\s*([\d,]+\.\d{2})', text, re.IGNORECASE)
        if m: return float(m.group(1).replace(',', ''))
        m = re.search(r'Fare\s+total\s*\$\s*([\d,]+\.\d{2})', text, re.IGNORECASE)
        return float(m.group(1).replace(',', '')) if m else None

    def _fare(self, text):
        m = re.search(r'Trip\s+fare\s+([€$£¥]?\s*[\d,]+\.\d{2})', text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    def _tip(self, text):
        m = re.search(r'\bTip\b\s+([€$£¥]?\s*[\d,]+\.\d{2})', text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    def _passenger(self, text):
        m = re.search(r'Thanks\s+for\s+(?:tipping|riding|using\s+Uber)[,\s]+([A-Za-z]+)', text, re.IGNORECASE)
        return m.group(1).strip() if m else None

    def _desc(self, text):
        m = re.search(r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{1,2},\s*\d{4}\b', text, re.IGNORECASE)
        return f'Uber Trip - {m.group(0)}' if m else 'Uber Trip'


# ── Shared date helper ────────────────────────────────────────────────────────
def _pd(s):
    try: return dparser.parse(s).strftime('%Y-%m-%d')
    except: return s.strip()


PARSERS = {
    'amtrak':   AmtrakParser(),
    'marriott': MarriottParser(),
    'uber':     UberParser(),
}
print('✅ Parsers ready: Amtrak | Marriott | Uber')

✅ Parsers ready: Amtrak | Marriott | Uber


## 📊 Step 6 — Excel / CSV Writer
Writes parsed receipts to a formatted Excel workbook and a flat CSV file.

In [33]:
import csv
from datetime import datetime as dt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

COMMON_COLS = [
    ('Vendor',       'vendor'),
    ('Date',         'date'),
    ('Amount ($)',   'amount'),
    ('Description',  'description'),
]
EXTRA_COLS = {
    'Amtrak':   [('Reservation #','reservation'),('Ticket #','ticket_number'),('Passenger','passenger'),('Purchase Date','purchase_date')],
    'Marriott': [('Check-Out','checkout_date'),('Nights','nights'),('Guest #','guest_number'),('Room #','room_number'),('Bonvoy #','bonvoy_number'),('Guest Name','guest_name'),('Charges','charge_summary')],
    'Uber':     [('Trip Fare','trip_fare'),('Tip','tip'),('Passenger','passenger')],
}
COLOURS = {'Amtrak':'215FA8','Marriott':'B11116','Uber':'000000','Summary':'2E4057'}


def write_receipts(records, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    ts         = dt.now().strftime('%Y%m%d_%H%M%S')
    excel_path = os.path.join(output_dir, f'travel_receipts_{ts}.xlsx')
    csv_path   = os.path.join(output_dir, f'travel_receipts_{ts}.csv')
    _write_excel(records, excel_path)
    _write_csv(records, csv_path)
    return excel_path, csv_path


def _write_excel(records, path):
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    vendors = sorted({r.get('vendor','?') for r in records})
    for v in vendors:
        rows = [r for r in records if r.get('vendor') == v]
        cols = COMMON_COLS + EXTRA_COLS.get(v, [])
        _add_sheet(wb, v, rows, cols, COLOURS.get(v,'444444'))
    if records:
        _add_sheet(wb, 'Summary', records, COMMON_COLS, COLOURS['Summary'])
        wb.move_sheet('Summary', offset=-len(vendors))
    wb.save(path)


def _add_sheet(wb, name, rows, cols, colour):
    ws = wb.create_sheet(title=name)
    hf = Font(bold=True, color='FFFFFF')
    hb = PatternFill('solid', fgColor=colour)
    ha = Alignment(horizontal='center', vertical='center')
    bd = Border(bottom=Side(style='thin', color='CCCCCC'), right=Side(style='thin', color='CCCCCC'))
    for ci, (hdr, _) in enumerate(cols, 1):
        c = ws.cell(row=1, column=ci, value=hdr)
        c.font, c.fill, c.alignment = hf, hb, ha
    ws.row_dimensions[1].height = 20
    for ri, rec in enumerate(rows, 2):
        for ci, (_, key) in enumerate(cols, 1):
            val = rec.get(key)
            c   = ws.cell(row=ri, column=ci, value=val)
            c.border = bd
            if key == 'amount' and val is not None:
                c.number_format = '"$"#,##0.00'
    for ci, (hdr, key) in enumerate(cols, 1):
        max_len = max([len(hdr)] + [len(str(r.get(key) or '')) for r in rows])
        ws.column_dimensions[get_column_letter(ci)].width = min(max_len + 4, 50)
    amt_col = next((i+1 for i,(_, k) in enumerate(cols) if k == 'amount'), None)
    if amt_col and rows:
        tr = len(rows) + 2
        tc = ws.cell(row=tr, column=amt_col)
        tc.value         = f'=SUM({get_column_letter(amt_col)}2:{get_column_letter(amt_col)}{tr-1})'
        tc.font          = Font(bold=True)
        tc.number_format = '"$"#,##0.00'
        ws.cell(row=tr, column=1, value='TOTAL').font = Font(bold=True)
    ws.freeze_panes = 'A2'


def _write_csv(records, path):
    if not records: return
    priority   = [k for _, k in COMMON_COLS]
    all_keys   = priority + [k for r in records for k in r if k not in priority]
    fieldnames = list(dict.fromkeys(all_keys))
    with open(path, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader(); w.writerows(records)


print('✅ Excel/CSV writer ready.')

✅ Excel/CSV writer ready.


## ☁️ Step 7 — Google Drive Uploader
Uploads files into your **VibeCodingExperiments → TravelReceipts** folder.

In [34]:
import mimetypes
from googleapiclient.http import MediaFileUpload


class DriveUploader:
    def __init__(self, credentials, root_folder_id):
        self.service          = build('drive', 'v3', credentials=credentials)
        self._root_id         = root_folder_id
        self._subfolder_cache = {}
        print(f'  📁 Drive root folder id: {root_folder_id}')

    def upload(self, local_path, subfolder=None):
        parent_id = self._get_or_create_subfolder(subfolder) if subfolder else self._root_id
        filename  = os.path.basename(local_path)
        mime_type = mimetypes.guess_type(local_path)[0] or 'application/octet-stream'
        existing  = self._find_file(filename, parent_id)
        if existing:
            print(f'  ⏭️  Already exists, skipping: {filename}')
            return existing
        created = self.service.files().create(
            body={'name': filename, 'parents': [parent_id]},
            media_body=MediaFileUpload(local_path, mimetype=mime_type, resumable=True),
            fields='id,name'
        ).execute()
        print(f'  ☁️  Uploaded: {created["name"]}  (id={created["id"]})')
        return created['id']

    def _get_or_create_subfolder(self, name):
        if name not in self._subfolder_cache:
            existing = self._find_folder(name, self._root_id)
            if existing:
                self._subfolder_cache[name] = existing
            else:
                folder = self.service.files().create(
                    body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [self._root_id]},
                    fields='id,name'
                ).execute()
                print(f'  📂 Created subfolder: {folder["name"]}')
                self._subfolder_cache[name] = folder['id']
        return self._subfolder_cache[name]

    def _find_folder(self, name, parent_id):
        return self._query(f"name='{name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed=false")

    def _find_file(self, name, parent_id):
        return self._query(f"name='{name}' and '{parent_id}' in parents and trashed=false")

    def _query(self, q):
        resp  = self.service.files().list(q=q, spaces='drive', fields='files(id,name)', pageSize=1).execute()
        files = resp.get('files', [])
        return files[0]['id'] if files else None


print('✅ DriveUploader ready.')

✅ DriveUploader ready.


---
## 🚀 Step 8 — Run Everything
This single cell runs all three tasks:
- **Task 1:** Scan Gmail → download receipts → upload raw files to Drive
- **Task 2:** Parse Amtrak receipts
- **Task 3:** Parse Marriott receipts (+ Uber)
- **Export:** Write Excel + CSV → upload to Drive

In [35]:
print('=' * 60)
print('  ✈️  Travel Receipt Automation')
print('=' * 60)

# ── Task 1: Scan Gmail ───────────────────────────────────────────────────────
print(f'\n📧 TASK 1 — Gmail scan ({START_MONTH} → {END_MONTH})')
print('-' * 60)

scanner = GmailScanner(creds)
downloaded = scanner.download_receipts(
    start_month  = START_MONTH,
    end_month    = END_MONTH,
    download_dir = DOWNLOAD_DIR,
    vendors      = VENDORS,
    queries      = GMAIL_QUERIES,
)
print(f'\n✅ Downloaded {len(downloaded)} file(s) from Gmail.')

# Upload raw receipts to Drive
if downloaded:
    print('\n☁️  Uploading raw receipts to Drive…')
    uploader = DriveUploader(creds, DRIVE_FOLDER_ID)
    for rec in downloaded:
        subfolder = rec.get('vendor', 'Other').capitalize()
        rec['drive_file_id'] = uploader.upload(rec['local_path'], subfolder=subfolder)
    print('✅ Raw receipts uploaded to Drive.')

# ── Tasks 2 & 3: Parse receipts ──────────────────────────────────────────────
print('\n🔍 TASKS 2 & 3 — Parsing receipts')
print('-' * 60)

parsed_records = []
for rec in downloaded:
    vendor     = rec.get('vendor', '').lower()
    local_path = rec.get('local_path', '')
    parser     = PARSERS.get(vendor)
    if not parser:
        print(f'  ⏭️  No parser for vendor "{vendor}" — skipping')
        continue
    if not local_path or not os.path.exists(local_path):
        print(f'  ⚠️  File not found: {local_path}')
        continue
    try:
        result = parser.parse(local_path)
        print(f'  ✅ [{vendor:10s}] {result.get("date","?"):12s}  ${result.get("amount") or 0:>8.2f}  {str(result.get("description",""))[:45]}')
        parsed_records.append(result)
    except Exception as e:
        print(f'  ❌ Error parsing {os.path.basename(local_path)}: {e}')

print(f'\n✅ Parsed {len(parsed_records)} receipt(s) successfully.')

# ── Export Excel + CSV ───────────────────────────────────────────────────────
if parsed_records:
    print('\n📊 Writing Excel + CSV…')
    excel_path, csv_path = write_receipts(parsed_records, OUTPUT_DIR)
    print(f'  📄 Excel : {excel_path}')
    print(f'  📄 CSV   : {csv_path}')

    # Upload Excel and CSV to Drive root TravelReceipts folder
    print('\n☁️  Uploading Excel & CSV to Drive…')
    uploader.upload(excel_path)   # goes to TravelReceipts root
    uploader.upload(csv_path)

    print('\n' + '=' * 60)
    print('  🎉 Done!')
    print(f'  Excel and CSV are in your Drive:')
    print(f'  My Drive → VibeCodingExperiments → TravelReceipts')
    print('=' * 60)
else:
    print('\n⚠️  No records parsed — check that receipts exist in the date range.')

  ✈️  Travel Receipt Automation

📧 TASK 1 — Gmail scan (2026-01 → 2026-04)
------------------------------------------------------------

📧 Searching AMTRAK receipts…
   Query: from:etickets@amtrak.com subject:"SALES RECEIPT" after:2026/01/01 before:2026/04/30


HttpError: <HttpError 403 when requesting https://gmail.googleapis.com/gmail/v1/users/me/messages?q=from%3Aetickets%40amtrak.com+subject%3A%22SALES+RECEIPT%22+after%3A2026%2F01%2F01+before%3A2026%2F04%2F30&maxResults=100&alt=json returned "Request had insufficient authentication scopes.". Details: "[{'message': 'Insufficient Permission', 'domain': 'global', 'reason': 'insufficientPermissions'}]">

---
## 👀 Step 9 — Preview Results (Optional)
Quickly view the parsed data as a table inside the notebook.

In [ ]:
import pandas as pd

if 'parsed_records' not in locals():
    parsed_records = []

if parsed_records:
    df = pd.DataFrame(parsed_records)[['vendor','date','amount','description']]
    df['amount'] = df['amount'].apply(lambda x: f'${x:,.2f}' if x else 'N/A')
    df.columns   = ['Vendor','Date','Amount','Description']
    display(df)
    print(f'\nTotal records: {len(df)}')
else:
    print('No parsed records to display.')